# Baseline Top-K Image Retrieval — Colab Bridge

This notebook is the **single entry point** for running Part 1 on Google Colab.
All model logic lives in the cloned repository; this file only handles
environment setup, configuration, and orchestration.

**Output** → `outputs/<encoder>/` on Colab, then synced to Google Drive  
**Consumed by** → Part 2 CNN Reranking (loads embeddings + Top-20 indices)

```
query_paths[i]  +  gallery_paths[topk_indices[i, j]]  →  (query, candidate) pair
label = int(query_labels[i] == gallery_labels[topk_indices[i, j]])
```

In [1]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
!pip install -q kaggle
!pip install -q timm
!pip install -q git+https://github.com/openai/CLIP.git

  Preparing metadata (setup.py) ... done


In [2]:
# ── 2. Mount Google Drive (outputs will be copied here at the end) ────────────
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ── 3. Clone project repository and add to Python path ────────────────────────
REPO_URL  = 'https://github.com/Syndr0/DL_Project.git'
REPO_DIR  = '/content/DL_Project'

!git clone {REPO_URL} {REPO_DIR} -q

import sys
sys.path.insert(0, REPO_DIR)
print('Repository cloned and added to sys.path.')

fatal: destination path '/content/DL_Project' already exists and is not an empty directory.
Repository cloned and added to sys.path.


In [4]:
# ── 4. Download Kaggle dataset ────────────────────────────────────────────────
# Upload kaggle.json: Kaggle → Settings → API → Create New Token
from google.colab import files
files.upload()   # select kaggle.json

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d iamsouravbanerjee/animal-image-dataset-90-different-animals \
       -p /content/data --unzip -q
print('Dataset ready.')

Saving kaggle.json to kaggle (6).json
Dataset URL: https://www.kaggle.com/datasets/iamsouravbanerjee/animal-image-dataset-90-different-animals
License(s): other
Dataset ready.


In [5]:
# ── 5. Imports ────────────────────────────────────────────────────────────────
import os, random, time
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

from retrieval_engine import (
    build_encoder, fine_tune, extract_all, save_outputs,
    save_metrics_json, submit, build_submission,
)
from baseline.utils.dataset import build_dataset_split
from baseline.utils.metrics import evaluate, precision_at_k

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

Device: cuda


In [6]:
# ── 6. Build dataset split (70 % gallery / 30 % query, per class) ─────────────
DATA_DIR = '/content/data/animals/animals'

train_paths, train_labels, query_paths, query_labels, classes = \
    build_dataset_split(DATA_DIR, train_ratio=0.7)

print(f'Gallery : {len(train_paths)} images')
print(f'Query   : {len(query_paths)} images')
print(f'Classes : {len(classes)}')

Gallery : 3780 images
Query   : 1620 images
Classes : 90


## Primary Encoder 1 — CLIP ViT-B/32
Strong zero-shot baseline. 512-d embeddings.

In [7]:
# ── 7a. Config ────────────────────────────────────────────────────────────────
BACKBONE1  = 'clip'
VARIANT1   = 'ViT-B/32'
POOL1      = 'gap'    # ignored for CLIP
FINE_TUNE1 = False    # set True to run proxy-classification fine-tuning
FT_EPOCHS  = 10

# ── 7b. Build encoder ─────────────────────────────────────────────────────────
model1, encode1, feat_dim1, tfm1, fwd1 = build_encoder(BACKBONE1, VARIANT1, POOL1)
print(f'feat_dim = {feat_dim1}')

# ── 7c. Optional fine-tuning ──────────────────────────────────────────────────
if FINE_TUNE1:
    print('Fine-tuning CLIP ...')
    t0 = time.time()
    fine_tune(model1, fwd1, feat_dim1, len(classes),
              train_paths, train_labels, tfm1,
              epochs=FT_EPOCHS, backbone=BACKBONE1)
    print(f'Fine-tuning done in {time.time()-t0:.1f}s')

# ── 7d. Extract embeddings ────────────────────────────────────────────────────
t0 = time.time()
gallery_embs1 = extract_all(train_paths, encode1)
query_embs1   = extract_all(query_paths,  encode1)
print(f'Extraction done in {time.time()-t0:.1f}s')

# ── 7e. Evaluate ──────────────────────────────────────────────────────────────
scores1     = evaluate(query_embs1, gallery_embs1, query_labels, train_labels)
precision1  = precision_at_k(query_embs1, gallery_embs1, query_labels, train_labels)
tag1 = f'CLIP ViT-B/32{" [FT]" if FINE_TUNE1 else ""}'
print(f'\n{tag1}')
for k, v in scores1.items():
    print(f'  Recall@{k:2d}   : {v:.4f}    Precision@{k}: {precision1[k]:.4f}')

save_metrics_json(tag1, scores1, precision1, pooling_type=POOL1)

# ── 7f. Save outputs for Part 2 ───────────────────────────────────────────────
out_tag1 = f'{REPO_DIR}/outputs/clip_ViT-B-32_gap{"_ft" if FINE_TUNE1 else ""}'
save_outputs(
    out_tag1,
    gallery_embs1, query_embs1,
    train_paths, query_paths,
    train_labels, query_labels,
    classes,
    topk_k=20,
    eval_scores=scores1,
    precision_scores=precision1,
    metadata={'backbone': BACKBONE1, 'variant': VARIANT1,
              'pool': POOL1, 'fine_tuned': FINE_TUNE1},
)

feat_dim = 512


Extracting embeddings: 100%|██████████| 26/26 [00:26<00:00,  1.03s/it]


Extraction done in 90.9s

CLIP ViT-B/32
  Recall@ 1   : 0.9210    Precision@1: 0.9210
  Recall@ 5   : 0.9870    Precision@5: 0.8736
  Recall@10   : 0.9938    Precision@10: 0.8336
Metrics saved → results_animals/CLIP_ViT-B-32_metrics_20260417_031529.json
Outputs saved to /content/DL_Project/outputs/clip_ViT-B-32_gap/
  gallery_embeddings : torch.Size([3780, 512])
  query_embeddings   : torch.Size([1620, 512])
  topk_indices       : torch.Size([1620, 20])  (Top-20)
  Recall@ 1          : 0.9210
  Recall@ 5          : 0.9870
  Recall@10          : 0.9938
  Precision@ 1       : 0.9210
  Precision@ 5       : 0.8736
  Precision@10       : 0.8336


## Primary Encoder 2 — ResNet-50 GAP


In [8]:
# ── 8a. Config ────────────────────────────────────────────────────────────────
BACKBONE2  = 'resnet'
VARIANT2   = '50'
POOL2      = 'gap'
FINE_TUNE2 = False

# ── 8b. Build + extract + evaluate + save ─────────────────────────────────────
model2, encode2, feat_dim2, tfm2, fwd2 = build_encoder(BACKBONE2, VARIANT2, POOL2)
print(f'feat_dim = {feat_dim2}')

if FINE_TUNE2:
    print('Fine-tuning ResNet-50 (layer4 + MLP head) ...')
    t0 = time.time()
    fine_tune(model2, fwd2, feat_dim2, len(classes),
              train_paths, train_labels, tfm2,
              epochs=FT_EPOCHS, backbone=BACKBONE2)
    print(f'Fine-tuning done in {time.time()-t0:.1f}s')

t0 = time.time()
gallery_embs2 = extract_all(train_paths, encode2)
query_embs2   = extract_all(query_paths,  encode2)
print(f'Extraction done in {time.time()-t0:.1f}s')

scores2    = evaluate(query_embs2, gallery_embs2, query_labels, train_labels)
precision2 = precision_at_k(query_embs2, gallery_embs2, query_labels, train_labels)
tag2 = f'ResNet-50 GAP{" [FT]" if FINE_TUNE2 else ""}'
print(f'\n{tag2}')
for k, v in scores2.items():
    print(f'  Recall@{k:2d}   : {v:.4f}    Precision@{k}: {precision2[k]:.4f}')

save_metrics_json(tag2, scores2, precision2, pooling_type=POOL2)

out_tag2 = f'{REPO_DIR}/outputs/resnet_50_gap{"_ft" if FINE_TUNE2 else ""}'
save_outputs(
    out_tag2,
    gallery_embs2, query_embs2,
    train_paths, query_paths,
    train_labels, query_labels,
    classes,
    topk_k=20,
    eval_scores=scores2,
    precision_scores=precision2,
    metadata={'backbone': BACKBONE2, 'variant': VARIANT2,
              'pool': POOL2, 'fine_tuned': FINE_TUNE2},
)

feat_dim = 2048


Extracting embeddings: 100%|██████████| 26/26 [00:24<00:00,  1.07it/s]


Extraction done in 81.9s

ResNet-50 GAP
  Recall@ 1   : 0.9037    Precision@1: 0.9037
  Recall@ 5   : 0.9747    Precision@5: 0.8551
  Recall@10   : 0.9858    Precision@10: 0.8262
Metrics saved → results_animals/ResNet-50_GAP_metrics_20260417_031653.json
Outputs saved to /content/DL_Project/outputs/resnet_50_gap/
  gallery_embeddings : torch.Size([3780, 2048])
  query_embeddings   : torch.Size([1620, 2048])
  topk_indices       : torch.Size([1620, 20])  (Top-20)
  Recall@ 1          : 0.9037
  Recall@ 5          : 0.9747
  Recall@10          : 0.9858
  Precision@ 1       : 0.9037
  Precision@ 5       : 0.8551
  Precision@10       : 0.8262


## Primary Encoder 3 — GoogLeNet base GeM
Classic inception architecture with Generalized Mean Pooling. 1024-d embeddings.

In [10]:
# ── GoogLeNet base GeM Config ────────────────────────────────────────────────
BACKBONE4  = 'googlenet'
VARIANT4   = 'base'
POOL4      = 'gem'
FINE_TUNE4 = False

# ── Build + extract + evaluate + save ─────────────────────────────────────────
model4, encode4, feat_dim4, tfm4, fwd4 = build_encoder(BACKBONE4, VARIANT4, POOL4)
print(f'feat_dim = {feat_dim4}')

if FINE_TUNE4:
    print('Fine-tuning GoogLeNet ...')
    t0 = time.time()
    fine_tune(model4, fwd4, feat_dim4, len(classes),
              train_paths, train_labels, tfm4,
              epochs=FT_EPOCHS, backbone=BACKBONE4)
    print(f'Fine-tuning done in {time.time()-t0:.1f}s')

t0 = time.time()
gallery_embs4 = extract_all(train_paths, encode4)
query_embs4   = extract_all(query_paths,  encode4)
print(f'Extraction done in {time.time()-t0:.1f}s')

scores4    = evaluate(query_embs4, gallery_embs4, query_labels, train_labels)
precision4 = precision_at_k(query_embs4, gallery_embs4, query_labels, train_labels)
tag4 = f'GoogLeNet base GeM{" [FT]" if FINE_TUNE4 else ""}'
print(f'\n{tag4}')
for k, v in scores4.items():
    print(f'  Recall@{k:2d}   : {v:.4f}    Precision@{k}: {precision4[k]:.4f}')

save_metrics_json(tag4, scores4, precision4, pooling_type=POOL4)

out_tag4 = f'{REPO_DIR}/outputs/googlenet_base_gem{"_ft" if FINE_TUNE4 else ""}'
save_outputs(
    out_tag4,
    gallery_embs4, query_embs4,
    train_paths, query_paths,
    train_labels, query_labels,
    classes,
    topk_k=20,
    eval_scores=scores4,
    precision_scores=precision4,
    metadata={'backbone': BACKBONE4, 'variant': VARIANT4,
              'pool': POOL4, 'fine_tuned': FINE_TUNE4},
)

feat_dim = 1024


Extracting embeddings: 100%|██████████| 26/26 [00:21<00:00,  1.21it/s]


Extraction done in 71.7s

GoogLeNet base GeM
  Recall@ 1   : 0.8840    Precision@1: 0.8840
  Recall@ 5   : 0.9494    Precision@5: 0.8000
  Recall@10   : 0.9691    Precision@10: 0.7551
Metrics saved → results_animals/GoogLeNet_base_GeM_metrics_20260417_031953.json
Outputs saved to /content/DL_Project/outputs/googlenet_base_gem/
  gallery_embeddings : torch.Size([3780, 1024])
  query_embeddings   : torch.Size([1620, 1024])
  topk_indices       : torch.Size([1620, 20])  (Top-20)
  Recall@ 1          : 0.8840
  Recall@ 5          : 0.9494
  Recall@10          : 0.9691
  Precision@ 1       : 0.8840
  Precision@ 5       : 0.8000
  Precision@10       : 0.7551


In [11]:
# ── 9. Sync outputs to Google Drive ──────────────────────────────────────────
DRIVE_OUT = '/content/drive/MyDrive/dl_project_outputs'
!mkdir -p {DRIVE_OUT}
!cp -r {REPO_DIR}/outputs/. {DRIVE_OUT}/
print(f'Outputs synced to Drive: {DRIVE_OUT}')

Outputs synced to Drive: /content/drive/MyDrive/dl_project_outputs


In [ ]:
# ── 10. Visualisation ─────────────────────────────────────────────────────────────────────
import torch.nn.functional as F

def show_retrieval(q_idx, gallery_embs, query_embs, g_paths, q_paths,
                   g_labels, q_labels, title='', k=5):
    sims = gallery_embs @ query_embs[q_idx]
    topk_vals, topk_idx = torch.topk(sims, k)
    fig, axes = plt.subplots(1, k + 1, figsize=(3 * (k + 1), 3))
    axes[0].imshow(Image.open(q_paths[q_idx]))
    axes[0].set_title('Query'); axes[0].axis('off')
    for i, (idx, s) in enumerate(zip(topk_idx.numpy(), topk_vals.numpy())):
        ok = g_labels[idx] == q_labels[q_idx]
        axes[i + 1].imshow(Image.open(g_paths[idx]))
        axes[i + 1].set_title(f"{'V' if ok else 'X'} {s:.3f}",
                               color='green' if ok else 'red', fontsize=10)
        axes[i + 1].axis('off')
    plt.suptitle(title, y=1.02)
    plt.tight_layout(); plt.show()

random.seed(0)
sample_qs = random.sample(range(len(query_paths)), 3)

for tag, g_embs, q_embs in [
    ('CLIP ViT-B/32',      gallery_embs1, query_embs1),
    ('ResNet-50 GAP',      gallery_embs2, query_embs2),
    ('GoogLeNet base GeM', gallery_embs4, query_embs4),
]:
    print(f'=== {tag} ===')
    for q in sample_qs:
        show_retrieval(q, g_embs, q_embs,
                       train_paths, query_paths, train_labels, query_labels,
                       title=f'{tag} | Query: {classes[query_labels[q]]}')

---
## Stanford Online Products — Cosine & L2 Ranking (Top-50)

Dataset: [liucong12601/stanford-online-products-dataset](https://www.kaggle.com/datasets/liucong12601/stanford-online-products-dataset)  
~120K product images across 22,634 classes. Only the **test split** (11,316 classes, ~60K images) is used.

**Split**: same 70/30 per-class strategy as the animal dataset — first 70% of each class (sorted by filename) → gallery, remaining 30% → query (~42K gallery, ~18K query).

Each query retrieves **two independent Top-50 lists**:
- **Cosine**: ranked by cosine similarity (dot product on L2-normalised vectors)
- **L2**: ranked by Euclidean distance (ascending)

In [ ]:
# ── SOP-0. Pull latest code and reload modules ───────────────────────────────
import sys
REPO_DIR = '/content/DL_Project'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

!cd /content/DL_Project && git pull -q

import importlib, baseline.utils.dataset, baseline.utils.metrics, retrieval_engine
importlib.reload(baseline.utils.dataset)
importlib.reload(baseline.utils.metrics)
importlib.reload(retrieval_engine)

from baseline.utils.dataset import build_stanford_split
from baseline.utils.metrics import evaluate, precision_at_k, evaluate_l2, precision_at_k_l2
from retrieval_engine       import extract_all, save_outputs_dual, save_metrics_json
print('Modules reloaded.')


In [ ]:
# ── SOP-1. Download Stanford Online Products from Kaggle ─────────────────────
!kaggle datasets download -d liucong12601/stanford-online-products-dataset \
       -p /content/data/sop --unzip -q
print('Stanford Online Products dataset ready.')

import os
sop_candidates = [
    '/content/data/sop/Stanford_Online_Products',
    '/content/data/sop',
]
SOP_DIR = next(
    p for p in sop_candidates
    if os.path.exists(os.path.join(p, 'Ebay_train.txt'))
)
print(f'SOP root: {SOP_DIR}')

In [ ]:
# ── SOP-2. Build gallery / query split ───────────────────────────────────────
from baseline.utils.dataset import build_stanford_split

sop_gallery_paths, sop_gallery_labels, \
sop_query_paths,   sop_query_labels,   \
sop_classes = build_stanford_split(SOP_DIR)

print(f'Gallery : {len(sop_gallery_paths):,} images')
print(f'Query   : {len(sop_query_paths):,}  images')
print(f'Classes : {len(sop_classes):,}')

In [ ]:
# ── SOP-3. Imports ───────────────────────────────────────────────────────────
import os, time
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

from retrieval_engine import extract_all, save_outputs_dual, save_metrics_json
from baseline.utils.metrics import evaluate, precision_at_k, evaluate_l2, precision_at_k_l2

SOP_TOPK   = 50
SOP_K_LIST = (1, 5, 10, 50)

In [ ]:
# ── SOP-3b. Build encoders (re-initialise if animals section was skipped) ────
try:
    encode1
except NameError:
    import sys; sys.path.insert(0, REPO_DIR)
    from retrieval_engine import build_encoder
    model1, encode1, feat_dim1, tfm1, fwd1 = build_encoder('clip',      'ViT-B/32', 'gap')
    model2, encode2, feat_dim2, tfm2, fwd2 = build_encoder('resnet',    '50',       'gap')
    model4, encode4, feat_dim4, tfm4, fwd4 = build_encoder('googlenet', 'base',     'gem')
    print('Encoders initialised.')
else:
    print('Reusing encoders from animals section.')

### SOP Encoder 1 — CLIP ViT-B/32

In [ ]:
# ── SOP CLIP ViT-B/32 ──────────────────────────────────────────────────────────────
t0 = time.time()
sop_gallery_embs1 = extract_all(sop_gallery_paths, encode1, batch_size=128)
sop_query_embs1 = extract_all(sop_query_paths,   encode1, batch_size=128)
print(f'Extraction: {time.time()-t0:.1f}s')

sop_cos_scores_1 = evaluate(         sop_query_embs1, sop_gallery_embs1, sop_query_labels, sop_gallery_labels, k_list=SOP_K_LIST)
sop_l2_scores_1  = evaluate_l2(      sop_query_embs1, sop_gallery_embs1, sop_query_labels, sop_gallery_labels, k_list=SOP_K_LIST)
sop_cos_prec_1   = precision_at_k(   sop_query_embs1, sop_gallery_embs1, sop_query_labels, sop_gallery_labels, k_list=SOP_K_LIST)
sop_l2_prec_1    = precision_at_k_l2(sop_query_embs1, sop_gallery_embs1, sop_query_labels, sop_gallery_labels, k_list=SOP_K_LIST)

print(f'\nCLIP ViT-B/32')
header = '  '.join(f'{k:>7}' for k in SOP_K_LIST)
print(f'{"":22}  {header}')
for metric, sc, pr in [
    ('cosine', sop_cos_scores_1, sop_cos_prec_1),
    ('L2',     sop_l2_scores_1,  sop_l2_prec_1),
]:
    row_r = '  '.join(f'{sc[k]:>7.4f}' for k in SOP_K_LIST)
    row_p = '  '.join(f'{pr[k]:>7.4f}' for k in SOP_K_LIST)
    print(f'  Recall    ({metric:<6})  {row_r}')
    print(f'  Precision ({metric:<6})  {row_p}')

save_metrics_json('SOP CLIP ViT-B/32 cosine', sop_cos_scores_1, sop_cos_prec_1, out_dir='results_sop')
save_metrics_json('SOP CLIP ViT-B/32 l2',     sop_l2_scores_1,  sop_l2_prec_1,  out_dir='results_sop')

save_outputs_dual(
    f'{REPO_DIR}/outputs/sop_clip_ViT_B_32_gap',
    sop_gallery_embs1, sop_query_embs1,
    sop_gallery_paths, sop_query_paths,
    sop_gallery_labels, sop_query_labels,
    sop_classes,
    topk_k=SOP_TOPK,
    eval_cos=sop_cos_scores_1, eval_l2=sop_l2_scores_1,
    prec_cos=sop_cos_prec_1,   prec_l2=sop_l2_prec_1,
    metadata={'backbone': 'clip', 'variant': 'ViT-B-32', 'pool': 'gap', 'dataset': 'SOP'},
)

### SOP Encoder 2 — ResNet-50 GAP

In [ ]:
# ── SOP ResNet-50 GAP ──────────────────────────────────────────────────────────────
t0 = time.time()
sop_gallery_embs2 = extract_all(sop_gallery_paths, encode2, batch_size=128)
sop_query_embs2 = extract_all(sop_query_paths,   encode2, batch_size=128)
print(f'Extraction: {time.time()-t0:.1f}s')

sop_cos_scores_2 = evaluate(         sop_query_embs2, sop_gallery_embs2, sop_query_labels, sop_gallery_labels, k_list=SOP_K_LIST)
sop_l2_scores_2  = evaluate_l2(      sop_query_embs2, sop_gallery_embs2, sop_query_labels, sop_gallery_labels, k_list=SOP_K_LIST)
sop_cos_prec_2   = precision_at_k(   sop_query_embs2, sop_gallery_embs2, sop_query_labels, sop_gallery_labels, k_list=SOP_K_LIST)
sop_l2_prec_2    = precision_at_k_l2(sop_query_embs2, sop_gallery_embs2, sop_query_labels, sop_gallery_labels, k_list=SOP_K_LIST)

print(f'\nResNet-50 GAP')
header = '  '.join(f'{k:>7}' for k in SOP_K_LIST)
print(f'{"":22}  {header}')
for metric, sc, pr in [
    ('cosine', sop_cos_scores_2, sop_cos_prec_2),
    ('L2',     sop_l2_scores_2,  sop_l2_prec_2),
]:
    row_r = '  '.join(f'{sc[k]:>7.4f}' for k in SOP_K_LIST)
    row_p = '  '.join(f'{pr[k]:>7.4f}' for k in SOP_K_LIST)
    print(f'  Recall    ({metric:<6})  {row_r}')
    print(f'  Precision ({metric:<6})  {row_p}')

save_metrics_json('SOP ResNet-50 GAP cosine', sop_cos_scores_2, sop_cos_prec_2, out_dir='results_sop')
save_metrics_json('SOP ResNet-50 GAP l2',     sop_l2_scores_2,  sop_l2_prec_2,  out_dir='results_sop')

save_outputs_dual(
    f'{REPO_DIR}/outputs/sop_resnet_50_gap',
    sop_gallery_embs2, sop_query_embs2,
    sop_gallery_paths, sop_query_paths,
    sop_gallery_labels, sop_query_labels,
    sop_classes,
    topk_k=SOP_TOPK,
    eval_cos=sop_cos_scores_2, eval_l2=sop_l2_scores_2,
    prec_cos=sop_cos_prec_2,   prec_l2=sop_l2_prec_2,
    metadata={'backbone': 'resnet', 'variant': '50', 'pool': 'gap', 'dataset': 'SOP'},
)

### SOP Encoder 3 — GoogLeNet base GeM

In [ ]:
# ── SOP GoogLeNet base GeM ──────────────────────────────────────────────────────────────
t0 = time.time()
sop_gallery_embs4 = extract_all(sop_gallery_paths, encode4, batch_size=128)
sop_query_embs4 = extract_all(sop_query_paths,   encode4, batch_size=128)
print(f'Extraction: {time.time()-t0:.1f}s')

sop_cos_scores_4 = evaluate(         sop_query_embs4, sop_gallery_embs4, sop_query_labels, sop_gallery_labels, k_list=SOP_K_LIST)
sop_l2_scores_4  = evaluate_l2(      sop_query_embs4, sop_gallery_embs4, sop_query_labels, sop_gallery_labels, k_list=SOP_K_LIST)
sop_cos_prec_4   = precision_at_k(   sop_query_embs4, sop_gallery_embs4, sop_query_labels, sop_gallery_labels, k_list=SOP_K_LIST)
sop_l2_prec_4    = precision_at_k_l2(sop_query_embs4, sop_gallery_embs4, sop_query_labels, sop_gallery_labels, k_list=SOP_K_LIST)

print(f'\nGoogLeNet base GeM')
header = '  '.join(f'{k:>7}' for k in SOP_K_LIST)
print(f'{"":22}  {header}')
for metric, sc, pr in [
    ('cosine', sop_cos_scores_4, sop_cos_prec_4),
    ('L2',     sop_l2_scores_4,  sop_l2_prec_4),
]:
    row_r = '  '.join(f'{sc[k]:>7.4f}' for k in SOP_K_LIST)
    row_p = '  '.join(f'{pr[k]:>7.4f}' for k in SOP_K_LIST)
    print(f'  Recall    ({metric:<6})  {row_r}')
    print(f'  Precision ({metric:<6})  {row_p}')

save_metrics_json('SOP GoogLeNet base GeM cosine', sop_cos_scores_4, sop_cos_prec_4, out_dir='results_sop')
save_metrics_json('SOP GoogLeNet base GeM l2',     sop_l2_scores_4,  sop_l2_prec_4,  out_dir='results_sop')

save_outputs_dual(
    f'{REPO_DIR}/outputs/sop_googlenet_base_gem',
    sop_gallery_embs4, sop_query_embs4,
    sop_gallery_paths, sop_query_paths,
    sop_gallery_labels, sop_query_labels,
    sop_classes,
    topk_k=SOP_TOPK,
    eval_cos=sop_cos_scores_4, eval_l2=sop_l2_scores_4,
    prec_cos=sop_cos_prec_4,   prec_l2=sop_l2_prec_4,
    metadata={'backbone': 'googlenet', 'variant': 'base', 'pool': 'gem', 'dataset': 'SOP'},
)

In [ ]:
# ── SOP-4. Summary table ──────────────────────────────────────────────────────
hdrs = '  '.join(f'R@{k}' for k in SOP_K_LIST)
print(f'{"Model":<26} {"Metric":<8} {hdrs}')
print('-' * 70)
for label, cos_sc, l2_sc in [
    ('CLIP ViT-B/32',      sop_cos_scores_1, sop_l2_scores_1),
    ('ResNet-50 GAP',      sop_cos_scores_2, sop_l2_scores_2),
    ('GoogLeNet base GeM', sop_cos_scores_4, sop_l2_scores_4),
]:
    row_cos = '  '.join(f'{cos_sc[k]:.4f}' for k in SOP_K_LIST)
    row_l2  = '  '.join(f'{l2_sc[k]:.4f}'  for k in SOP_K_LIST)
    print(f'{label:<26} {"cosine":<8} {row_cos}')
    print(f'{"":26} {"L2":<8} {row_l2}')

In [ ]:
# ── SOP-5. Sync SOP outputs to Google Drive ─────────────────────────────────
import os
_repo  = globals().get('REPO_DIR',  '/content/DL_Project')
_drive = globals().get('DRIVE_OUT', '/content/drive/MyDrive/dl_project_outputs')
os.makedirs(_drive, exist_ok=True)
!cp -r {_repo}/outputs/sop_* {_drive}/ 2>/dev/null || echo 'No sop_ outputs found'
!cp -r results_sop {_drive}/ 2>/dev/null || echo 'No results_sop found'
print(f'SOP outputs synced to {_drive}')
